On commence par un cas de classification tabulaire avec une regression logisitique sur 5 datasets de UCI. 
Ensuite on s'intéresse à de la classification par Image et aux propriétées des classifieurs obtenus par Nll, PO(I) et PO(Beta^Star). 

In [1]:
# !pip install scikit-learn pandas matplotlib seaborn torch --quiet

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import load_wine, load_breast_cancer, load_iris, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import seaborn as sns
import matplotlib.pyplot as plt

import nll_to_po.training.loss as L
import nll_to_po.training.reward as R
import nll_to_po.models.dn_policy as Policy
from nll_to_po.training.utils import train_single_policy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)

sns.set_theme(style="whitegrid", font_scale=1.5)
sns.set_palette("colorblind")
sns.despine()

/opt/anaconda3/envs/nllpo/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


<Figure size 640x480 with 0 Axes>

In [2]:
def Load_uci(
    dataset="wine", test_size=0.2, val_size=0.2, batch_size=256, standardize=True
):
    if dataset == "wine":
        data = load_wine()
    if dataset == "wine":
        data = load_wine()
    elif dataset == "iris":
        data = load_iris()
    elif dataset == "breast_cancer":
        data = load_breast_cancer()
    elif dataset == "load_digits":
        data = load_digits()
    else:
        raise ValueError(f"Unknown dataset: {dataset}")
    X, y = data.data, data.target
    if standardize:
        scaler = StandardScaler().fit(X)
        X = scaler.transform(X)

    # train / (val+test)
    X_tr, X_tt, y_tr, y_tt = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=0
    )
    # split val from train
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_tr, y_tr, test_size=val_size, stratify=y_tr, random_state=0
    )
    # conversion en tensor type pour le donner au dataloader
    X_tr = torch.tensor(X_tr, dtype=torch.float32)
    X_val = torch.tensor(X_val, dtype=torch.float32)
    X_tt = torch.tensor(X_tt, dtype=torch.float32)
    y_tr = torch.tensor(y_tr, dtype=torch.long)
    y_val = torch.tensor(y_val, dtype=torch.long)
    y_tt = torch.tensor(y_tt, dtype=torch.long)

    tr_loader = DataLoader(
        TensorDataset(X_tr, y_tr, y_tr, torch.zeros_like(y_tr)),
        batch_size=batch_size,
        shuffle=True,
    )
    # on ne shuffle pas la val et le test
    val_loader = DataLoader(
        TensorDataset(X_val, y_val, y_val, torch.zeros_like(y_val)),
        batch_size=batch_size,
        shuffle=False,
    )
    tst_loader = DataLoader(
        TensorDataset(X_tt, y_tt, y_tt, torch.zeros_like(y_tt)),
        batch_size=batch_size,
        shuffle=False,
    )

    meta = {"input_dim": X.shape[1], "num_classes": len(np.unique(y))}
    return tr_loader, val_loader, tst_loader, meta




In [81]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import load_wine, load_iris, load_breast_cancer, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

from ucimlrepo import fetch_ucirepo

UCI_NAME_TO_ID = {
    "credit_default": 350,        # Default of Credit Card Clients
    "german_credit": 144,         # Statlog German Credit
    "polish_bankruptcy": 365,     # Polish Companies Bankruptcy
    "adult": 2,                  # Adult (si besoin)
    # Ajouts "difficiles" pour régression logistique
    "aps_failure": 421,           # APS Failure at Scania Trucks
    "secom": 179,                 # Semiconductor Manufacturing (SECOM)
    "bank_marketing": 222,        # Bank Marketing
    "australian_credit": 143,     # Statlog Australian Credit Approval
    "magic_gamma": 159,           # MAGIC Gamma Telescope
    "ionosphere": 52,             # Ionosphere
    "wilt": 285,                  # Wilt (diseased trees)
    "higgs": 280,                 # HIGGS
    "spambase": 94,               # (optionnel) Spambase texte
    "Yeast": 110,
    'poker': 158
}                 

def _ensure_numpy_X_y(X, y):
    # X peut être DataFrame ou ndarray ; y peut être Series/ndarray/obj
    if isinstance(X, pd.DataFrame):
        X = X.to_numpy()
    if isinstance(y, (pd.Series, pd.DataFrame)):
        y = y.to_numpy().ravel()

    # Encode labels non-numériques (ex: 'good'/'bad')
    if y.dtype.kind not in "iu":  # pas int/uint
        le = LabelEncoder()
        y = le.fit_transform(y.astype(str))
    return X, y.astype(np.int64, copy=False)

def load_uci(
    dataset="wine",
    test_size=0.2,
    val_size=0.2,
    batch_size=256,
    standardize=True,
    impute_missing=False,
    impute_strategy="median",
    random_state=0,
    uci_id=None,
):
    """
    Charge un dataset soit via scikit-learn (wine/iris/breast_cancer/load_digits),
    soit via UCI (ucimlrepo) si `uci_id` est fourni ou si `dataset` est dans UCI_NAME_TO_ID.
    Renvoie (train_loader, val_loader, test_loader, meta) avec le même format TensorDataset
    que ton code (X, y, y, zeros).
    """
    # --- 1) Sélection de la source des données ---
    if dataset in {"wine", "iris", "breast_cancer", "load_digits"} and uci_id is None:
        if dataset == "wine":
            data = load_wine()
        elif dataset == "iris":
            data = load_iris()
        elif dataset == "breast_cancer":
            data = load_breast_cancer()
        elif dataset == "load_digits":
            data = load_digits()
        X, y = data.data, data.target

    else:
        # UCI via ucimlrepo
        if uci_id is None:
            # essaie de mapper un alias connu, sinon lève une erreur claire
            if dataset in UCI_NAME_TO_ID:
                uci_id = UCI_NAME_TO_ID[dataset]
            else:
                raise ValueError(
                    f"Unknown dataset '{dataset}'. "
                    f"Use one of {{'wine','iris','breast_cancer','load_digits'}} "
                    f"or provide a valid UCI id via `uci_id`."
                )
        ds = fetch_ucirepo(id=uci_id)
        X = ds.data.features
        y = ds.data.targets
        if isinstance(y, pd.DataFrame) and y.shape[1] > 1:
            y = y.iloc[:, 0]

        X, y = _ensure_numpy_X_y(X, y)

    if impute_missing:
        imp = SimpleImputer(strategy=impute_strategy)
        X = imp.fit_transform(X)

    if standardize:
        scaler = StandardScaler().fit(X)
        X = scaler.transform(X)

    X_tr, X_tt, y_tr, y_tt = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=random_state
    )
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_tr, y_tr, test_size=val_size, stratify=y_tr, random_state=random_state
    )

    X_tr = torch.tensor(X_tr, dtype=torch.float32)
    X_val = torch.tensor(X_val, dtype=torch.float32)
    X_tt = torch.tensor(X_tt, dtype=torch.float32)
    y_tr = torch.tensor(y_tr, dtype=torch.long)
    y_val = torch.tensor(y_val, dtype=torch.long)
    y_tt = torch.tensor(y_tt, dtype=torch.long)

    tr_loader = DataLoader(
        TensorDataset(X_tr, y_tr, y_tr, torch.zeros_like(y_tr)),
        batch_size=batch_size,
        shuffle=True,
    )
    val_loader = DataLoader(
        TensorDataset(X_val, y_val, y_val, torch.zeros_like(y_val)),
        batch_size=batch_size,
        shuffle=False,
    )
    tst_loader = DataLoader(
        TensorDataset(X_tt, y_tt, y_tt, torch.zeros_like(y_tt)),
        batch_size=batch_size,
        shuffle=False,
    )

    meta = {"input_dim": X.shape[1], "num_classes": int(np.unique(y).size)}
    return tr_loader, val_loader, tst_loader, meta


Set up $\widehat{Y}$ c'est le one hot encoding des labels Y et je travaille avec les one hot encoding

In [4]:
@torch.no_grad()
def Estimate_trace_sigma_onehot(train_loader, num_classes: int) -> float:
    #modification pour qu'elle accetpe nimporte quel dataloader avec des labels en deuxieme position
    ys = []
    print("Estimating trace of label covariance matrix...")
    for _, yb, *rest in train_loader:
        ys.append(yb)
    print("Done")
    y = torch.cat(ys, dim=0)  # (N,)
    Y = F.one_hot(y, num_classes=num_classes).float()  # (N,C)
    Yc = Y - Y.mean(dim=0, keepdim=True)  # (N,C)
    Sigma = (Yc.T @ Yc) / max(Y.shape[0] - 1, 1)  # (C,C)
    return float(torch.trace(Sigma))

import torch

def _trace_from_counts(counts: torch.Tensor) -> float:
    """
    counts: 1D Long tensor of length C with class counts.
    """
    N = int(counts.sum().item())
    if N <= 1:
        return 0.0
    p2_sum = (counts.float() / N).pow(2).sum().item()
    return float((N / (N - 1)) * (1.0 - p2_sum))

@torch.no_grad()
def estimate_trace_sigma_onehot(train_loader, num_classes: int) -> float:
    print('in estimate_trace_sigma_onehot')
    """
    Fast: streams the loader once, accumulates class counts via bincount.
    Works with any DataLoader where labels are at index 1.
    Accepts integer labels or one-hot labels (NxC).
    """
    counts = torch.zeros(num_classes, dtype=torch.long)
    print("finished counts")
    print("len(dataset) =", len(train_loader.dataset))
    print("len(loader)  =", len(train_loader))  # nb de batches attendu

    for batch in train_loader:
        yb = batch[1]
        # If labels are one-hot, convert to class indices
        if yb.ndim == 2 and yb.size(1) == num_classes:
            yb = yb.argmax(dim=1)
        yb = yb.to(torch.long).flatten()
        counts += torch.bincount(yb, minlength=num_classes)
    return _trace_from_counts(counts)



def beta_star_from_data(train_loader, entropy_weight: float, num_classes: int) -> float:
    print('in beta_star_from_data')
    tr = estimate_trace_sigma_onehot(train_loader, num_classes=num_classes)
    return float(entropy_weight * num_classes / (2.0 * max(tr, 1e-12)))

In [5]:
@torch.no_grad
def evaluate_accuracy_multi_regression(
    policy: nn.Module, data_loader: DataLoader
) -> float:
    policy.eval()
    correct, total = 0, 0
    for xb, yb, *rest in data_loader:
        # yb est un vecteur [num_samples] a valeur dans les labes [0;1,...,C-1]
        _, prob_predit = policy(xb)
        # one_hot_pred = prob_predit.argmax(dim=-1)
        # print(f'prob shape {prob_predit.shape}')
        # print(f'one hot true shape {yb.shape}')
        y_prob = prob_predit.argmax(dim=-1)
        correct += (y_prob == yb).sum().item()
        total += yb.numel()
    if total == 1:
        return "One element in the dataloader"
    else:
        return correct / max(total, 1)

In [6]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix

@torch.no_grad()
def _collect_binary_scores_and_labels(policy: nn.Module,
                                      data_loader: DataLoader,
                                      device: str = "cpu"):
    """
    Returns:
      y_true: (N,) numpy array in {0,1}
      scores: (N,) numpy array of positive-class scores (probabilities)
    """
    policy.eval()
    ys, ss = [], []
    for xb, yb, *rest in data_loader:
        xb = xb.to(device)
        yb = yb.to(device).long().view(-1)
        _, out = policy(xb)

        # Normalize into positive-class probability
        if out.ndim == 1 or (out.ndim == 2 and out.size(1) == 1):
            out = out.view(-1)
            if (out.max() > 1) or (out.min() < 0):   # logits
                pos_prob = out.sigmoid()
            else:                                    # already prob
                pos_prob = out.clamp(1e-12, 1-1e-12)
        elif out.ndim == 2 and out.size(1) == 2:
            probs = out.softmax(dim=1) if (out.min() < 0 or out.max() > 1) else out
            pos_prob = probs[:, 1].clamp(1e-12, 1-1e-12)
        else:
            raise ValueError(f"Unexpected output shape {out.shape}")

        ys.append(yb.cpu())
        ss.append(pos_prob.cpu())

    y_true = torch.cat(ys, 0).numpy()
    scores = torch.cat(ss, 0).numpy()
    return y_true, scores


def evaluate_tpr_tnr_binary(policy, data_loader, threshold=0.5, device="cpu"):
    y_true, scores = _collect_binary_scores_and_labels(policy, data_loader, device)
    y_pred = (scores >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    TPR = tp / (tp + fn) if (tp + fn) else 0.0
    TNR = tn / (tn + fp) if (tn + fp) else 0.0
    FPR = 1 - TNR
    FNR = 1 - TPR
    ACC = (tp + tn) / max(tp + tn + fp + fn, 1)
    BAL_ACC = (TPR + TNR) / 2.0

    return dict(TP=tp, TN=tn, FP=fp, FN=fn,
                TPR=TPR, TNR=TNR, FPR=FPR, FNR=FNR,
                ACC=ACC, BAL_ACC=BAL_ACC)


def evaluate_roc_auc_binary(policy, data_loader, device="cpu"):
    y_true, scores = _collect_binary_scores_and_labels(policy, data_loader, device)
    fpr, tpr, thresholds = roc_curve(y_true, scores)
    auc = roc_auc_score(y_true, scores)
    return dict(fpr=fpr, tpr=tpr, thresholds=thresholds, auc=auc)


In [7]:
import numpy as np
import tensorflow_datasets 
from tensorflow_datasets.core.utils.lazy_imports_utils import pandas as pd
from tensorflow_datasets.core.utils.lazy_imports_utils import tensorflow as tf

human_labels_np_path = '/Users/gsinger/Reinforcment_Learning_Huawei/nll_to_po/notebook/CIFAR_NOISY_DATA/CIFAR-10_human.npy'
human_labels_csv_path = '/Users/gsinger/Reinforcment_Learning_Huawei/nll_to_po/notebook/CIFAR_NOISY_DATA/CIFAR-10_human_annotations.csv'

with tf.io.gfile.GFile(human_labels_np_path, "rb") as f:
  human_annotations = np.load(f, allow_pickle=True)

df = pd.DataFrame(human_annotations[()])

with tf.io.gfile.GFile(human_labels_csv_path, "w") as f:
  df.to_csv(f, index=False)


In [8]:
import numpy as np
import torch
from torch.utils.data import DataLoader, SubsetRandomSampler, Dataset
from torchvision import datasets, transforms
from PIL import Image
from typing import Optional, Dict

# CIFAR-10 normalization
_CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
_CIFAR10_STD  = (0.2470, 0.2435, 0.2616)

class CIFAR10NTrain(Dataset):
    """
    Wraps torchvision CIFAR-10 (train=True) and replaces labels by a provided array (len=50000).
    Keeps the image order intact.
    """
    def __init__(self, root: str, noisy_labels: np.ndarray, transform=None, download: bool = True):
        base = datasets.CIFAR10(root=root, train=True, download=download)
        assert len(base.data) == 50000 and len(noisy_labels) == 50000
        self.data = base.data                      # N x 32 x 32 x 3 (uint8)
        self.labels = noisy_labels.astype(np.int64)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx: int):
        img = Image.fromarray(self.data[idx])
        y = int(self.labels[idx])
        if self.transform is not None:
            img = self.transform(img)
        # Return placeholder tensors for mu and std to match the expected format
        return img, y, torch.zeros_like(torch.tensor(y, dtype=torch.float32)), torch.zeros_like(torch.tensor(y, dtype=torch.float32))

def _load_cifar10n_labels(
    human_labels_np_path: Optional[str] = None,
    human_labels_dict: Optional[Dict[str, np.ndarray]] = None,
) -> Dict[str, np.ndarray]:
    """
    Load the CIFAR-10N dict from a .npy file (allow_pickle=True) or accept a preloaded dict.
    Returns a dict with keys like: 'clean_label', 'aggre_label', 'worse_label', 'random_label1/2/3', ...
    """
    if human_labels_dict is None:
        assert human_labels_np_path is not None, "Provide human_labels_np_path or human_labels_dict."
        d = np.load(human_labels_np_path, allow_pickle=True)
        # Some .npy store a 0-d object array containing the dict
        if isinstance(d, np.ndarray) and d.dtype == object:
            human_labels_dict = d.item()
        elif isinstance(d, dict):
            human_labels_dict = d
        else:
            raise ValueError("Unexpected CIFAR-10N npy format; expected dict or object array with dict.")
    return human_labels_dict

def load_cifar10n_dataset(
    batch_size: int = 128,
    data_root: str = "./data",
    label_key: str = "random_label1",
    val_fraction: float = 0.2,
    seed: int = 0,
    human_labels_np_path: Optional[str] = None,
    human_labels_dict: Optional[Dict[str, np.ndarray]] = None,
    download: bool = False,
):
    """
    Returns (train_loader, val_loader, test_loader, meta) using CIFAR-10 images and CIFAR-10N labels.

    - Train/Val split is performed on the 50k training images (shuffle once with `seed`).
    - Labels for BOTH train and val come from `label_key` (noisy validation, like your MNIST helper).
      If you want a clean-val variant, ask and I’ll add a switch.
    - Test set is the standard clean CIFAR-10 test.
    """
    # --- transforms ---
    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(_CIFAR10_MEAN, _CIFAR10_STD),
    ])
    eval_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(_CIFAR10_MEAN, _CIFAR10_STD),
    ])

    # --- load CIFAR-10N label dict ---
    labels_dict = _load_cifar10n_labels(human_labels_np_path, human_labels_dict)

    # Support the common typo 'worst_label' -> 'worse_label'
    if label_key == "worst_label" and "worse_label" in labels_dict:
        label_key = "worse_label"

    assert label_key in labels_dict, f"{label_key=} not found. Available: {list(labels_dict.keys())}"
    noisy_labels = np.asarray(labels_dict[label_key])
    assert noisy_labels.shape == (50000,), "CIFAR-10N labels must be length 50000."

    dl_kwargs = dict(num_workers=0, pin_memory=False, persistent_workers=False)

    # --- datasets ---
    train_val_dataset = CIFAR10NTrain(root=data_root, noisy_labels=noisy_labels, transform=train_tf, download=download)
    test_dataset_base = datasets.CIFAR10(root=data_root, train=False, download=download, transform=eval_tf)

    # Create a dataset for the test set that also includes placeholder tensors
    class CIFAR10TestWithPlaceholders(Dataset):
        def __init__(self, base_dataset):
            self.base = base_dataset

        def __len__(self):
            return len(self.base)

        def __getitem__(self, idx):
            # Unpack all returned values and take the first two
            data = self.base[idx]
            img, y = data[0], data[1]
            return img, y, torch.zeros_like(torch.tensor(y, dtype=torch.float32)), torch.zeros_like(torch.tensor(y, dtype=torch.float32))

    test_dataset = CIFAR10TestWithPlaceholders(test_dataset_base)

    num_train = len(train_val_dataset)  # 50k
    indices = np.arange(num_train)
    split = int(val_fraction * num_train)
    rng = np.random.RandomState(seed)
    rng.shuffle(indices)
    val_idx, train_idx = indices[:split], indices[split:]

    train_loader = DataLoader(
        train_val_dataset, batch_size=batch_size, sampler=SubsetRandomSampler(train_idx), **dl_kwargs
    )
    # Use eval transforms for val (no aug). Create a shallow copy with eval_tf.
    # Easiest: re-instantiate a view of the dataset but with eval_tf for the same labels.
    # Modify to include placeholder tensors for mu and std
    val_dataset_evalview_base = CIFAR10NTrain(root=data_root, noisy_labels=noisy_labels, transform=eval_tf, download=False)
    val_dataset_evalview = CIFAR10TestWithPlaceholders(val_dataset_evalview_base)

    val_loader = DataLoader(
        val_dataset_evalview, batch_size=batch_size, sampler=SubsetRandomSampler(val_idx), **dl_kwargs
    )

    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, **dl_kwargs)

    meta = {
        "in_channels": 3,
        "num_classes": 10,
        "image_size": (32, 32),
        "mean": _CIFAR10_MEAN,
        "std": _CIFAR10_STD,
        "label_key": label_key,
    }
    return train_loader, val_loader, test_loader, meta

In [9]:
import numpy as np
import torch
from torch.utils.data import DataLoader, SubsetRandomSampler, Dataset
from torchvision import datasets, transforms
from PIL import Image
from typing import Optional, Dict

# CIFAR-10 normalization
_CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
_CIFAR10_STD  = (0.2470, 0.2435, 0.2616)

class CIFAR10NTrain(Dataset):
    """
    Wraps torchvision CIFAR-10 (train=True) and replaces labels by a provided array (len=50000).
    Keeps the image order intact.
    """
    def __init__(self, root: str, noisy_labels: np.ndarray, transform=None, download: bool = True):
        base = datasets.CIFAR10(root=root, train=True, download=download)
        assert len(base.data) == 50000 and len(noisy_labels) == 50000
        self.data = base.data                      # N x 32 x 32 x 3 (uint8)
        self.labels = noisy_labels.astype(np.int64)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx: int):
        img = Image.fromarray(self.data[idx])
        y = int(self.labels[idx])
        if self.transform is not None:
            img = self.transform(img)
        return img, y

def _load_cifar10n_labels(
    human_labels_np_path: Optional[str] = None,
    human_labels_dict: Optional[Dict[str, np.ndarray]] = None,
) -> Dict[str, np.ndarray]:
    """
    Load the CIFAR-10N dict from a .npy file (allow_pickle=True) or accept a preloaded dict.
    Returns a dict with keys like: 'clean_label', 'aggre_label', 'worse_label', 'random_label1/2/3', ...
    """
    if human_labels_dict is None:
        assert human_labels_np_path is not None, "Provide human_labels_np_path or human_labels_dict."
        d = np.load(human_labels_np_path, allow_pickle=True)
        if isinstance(d, np.ndarray) and d.dtype == object:
            human_labels_dict = d.item()
        elif isinstance(d, dict):
            human_labels_dict = d
        else:
            raise ValueError("Unexpected CIFAR-10N npy format; expected dict or object array with dict.")
    return human_labels_dict

def load_cifar10n_dataset(
    batch_size: int = 128,
    data_root: str = "./data",
    label_key: str = "random_label1",   
    val_fraction: float = 0.2,
    seed: int = 0,
    human_labels_np_path: Optional[str] = None,
    human_labels_dict: Optional[Dict[str, np.ndarray]] = None,
    download: bool = False,
):
    """
    Returns (train_loader, val_loader, test_loader, meta) using CIFAR-10 images and CIFAR-10N labels.

    - Train/Val split is performed on the 50k training images (shuffle once with `seed`).
    - Labels for BOTH train and val come from `label_key` (noisy validation, like your MNIST helper).
      If you want a clean-val variant, ask and I’ll add a switch.
    - Test set is the standard clean CIFAR-10 test.
    """
    # --- transforms ---
    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(_CIFAR10_MEAN, _CIFAR10_STD),
    ])
    eval_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(_CIFAR10_MEAN, _CIFAR10_STD),
    ])

    # --- load CIFAR-10N label dict ---
    labels_dict = _load_cifar10n_labels(human_labels_np_path, human_labels_dict)

    # Support the common typo 'worst_label' -> 'worse_label'
    if label_key == "worst_label" and "worse_label" in labels_dict:
        label_key = "worse_label"

    assert label_key in labels_dict, f"{label_key=} not found. Available: {list(labels_dict.keys())}"
    noisy_labels = np.asarray(labels_dict[label_key])
    assert noisy_labels.shape == (50000,), "CIFAR-10N labels must be length 50000."

    dl_kwargs = dict(num_workers=0, pin_memory=False, persistent_workers=False)

    # --- datasets ---
    train_val_dataset = CIFAR10NTrain(root=data_root, noisy_labels=noisy_labels, transform=train_tf, download=download)
    test_dataset = datasets.CIFAR10(root=data_root, train=False, download=download, transform=eval_tf)

    # --- split 80/20 like your MNIST helper ---
    num_train = len(train_val_dataset)  # 50k
    indices = np.arange(num_train)
    split = int(val_fraction * num_train)
    rng = np.random.RandomState(seed)
    rng.shuffle(indices)
    val_idx, train_idx = indices[:split], indices[split:]

    train_loader = DataLoader(
        train_val_dataset, batch_size=batch_size, sampler=SubsetRandomSampler(train_idx), **dl_kwargs
    )
    # Use eval transforms for val (no aug). Create a shallow copy with eval_tf.
    # Easiest: re-instantiate a view of the dataset but with eval_tf for the same labels.
    val_dataset_evalview = CIFAR10NTrain(root=data_root, noisy_labels=noisy_labels, transform=eval_tf, download=False)
    val_loader = DataLoader(
        val_dataset_evalview, batch_size=batch_size, sampler=SubsetRandomSampler(val_idx), **dl_kwargs
    )

    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, **dl_kwargs)

    meta = {
        "in_channels": 3,
        "num_classes": 10,
        "image_size": (32, 32),
        "mean": _CIFAR10_MEAN,
        "std": _CIFAR10_STD,
        "label_key": label_key,
    }
    return train_loader, val_loader, test_loader, meta


In [10]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, SubsetRandomSampler
import numpy as np
import torch

def load_mnist_dataset(batch_size: int = 128):
    transform = transforms.Compose([transforms.ToTensor()])  # (B,1,28,28) dans [0,1]
    train_dataset = datasets.MNIST(root="./data", train=True,  download=True, transform=transform)
    test_dataset  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

    # Split train -> train/val (80/20)
    num_train = len(train_dataset)
    indices = np.arange(num_train)
    split = int(0.2 * num_train)
    rng = np.random.RandomState(0)
    rng.shuffle(indices)
    val_idx, train_idx = indices[:split], indices[split:]

    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=SubsetRandomSampler(train_idx))
    val_loader   = DataLoader(train_dataset, batch_size=batch_size, sampler=SubsetRandomSampler(val_idx))
    test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

    meta = {"in_channels": 1, "num_classes": 10, "image_size": (28, 28)}
    return train_loader, val_loader, test_loader, meta


In [63]:
#SANS LES FP FN ROC AUC AJOUTS
def Run_one_dataset(
    dataset_name: str,
    n_updates: int = 10,
    n_experiments: int = 5,
    batch_size: int = 128,
    learning_rate: float = 1e-4,
    entropy_weight: float = 1e-3,
    type_of_data : str = "tabular",
    corrupted_inputs: bool = False,
):
    if type_of_data=="tabular":
        print(f"Running on tabular dataset: {dataset_name}")
        train_loader, val_loader, test_loader, meta = load_uci(
        dataset=dataset_name, batch_size=batch_size, standardize=True
        )
        C = meta["num_classes"]
        D_in = meta["input_dim"]
    elif type_of_data=="image" and dataset_name=="mnist":
        print(f"Running on {dataset_name}")
        train_loader, val_loader, test_loader, meta = load_mnist_dataset(
            batch_size=batch_size
        )
        C = meta["num_classes"]
        D_in = meta["in_channels"]
    elif type_of_data=="image" and dataset_name=="cifar10n":
        human_labels_np_path = "/Users/gsinger/Reinforcment_Learning_Huawei/nll_to_po/notebook/CIFAR_NOISY_DATA/CIFAR-10_human.npy"

        train_loader, val_loader, test_loader, meta = load_cifar10n_dataset(
            batch_size=128,
            data_root="./data",
            label_key="random_label2",            
            val_fraction=0.2,
            seed=0,
            human_labels_np_path=human_labels_np_path,
        )
        C=meta["num_classes"]
        D_in=meta["in_channels"]

    print('finished to load datasets')
    beta_star = beta_star_from_data(
        train_loader, entropy_weight=entropy_weight, num_classes=C
    )
    print(f"Estimated beta_star: {beta_star:.4e}")
    # beta_list = [1, beta_star]
    #beta_stars = np.logspace(-2, 2, 10)  # de 0.01 à 100
    #beta_list = [beta_star] + beta_stars.tolist()      
    beta_list = [1, beta_star]

    curves = []
    tests = []

    for rep in range(n_experiments):
        # policy = MLPClassifier(D_in, C)
        if type_of_data=="tabular":
            policy = Policy.MulticlassLogisticRegression(D_in, C)
        elif type_of_data=="image":
            #policy = Policy.CNNClassifier(D_in, C)
            policy=Policy.ResNet18Vanilla(num_classes=C, in_channels=D_in)
        loss_fn = L.NLL_Classification()
        print("start training NLL") 
        trained_policy, train_metrics, val_metrics, _, _ = train_single_policy(
            policy=policy,
            train_dataloader=train_loader,
            val_dataloader=val_loader,
            loss_function=loss_fn,
            n_updates=n_updates,
            learning_rate=learning_rate,
            wandb_run=None,
            tensorboard_writer=None,
            logger=None,
            scheduler_patience=20, early_stopping_patience=n_updates,device=torch.device('cpu')
        )

        df_tr = (
            pd.DataFrame(train_metrics).reset_index().rename(columns={"index": "epoch"})
        )
        df_tr["split"] = "train"
        df_tr["method"] = "NLL"
        df_tr["beta"] = np.nan
        df_tr["rep"] = rep
        df_val = (
            pd.DataFrame(val_metrics).reset_index().rename(columns={"index": "epoch"})
        )
        df_val["split"] = "val"
        df_val["method"] = "NLL"
        df_val["beta"] = np.nan
        df_val["rep"] = rep
        curves += [df_tr, df_val]

        # test_acc = evaluate_accuracy(policy, test_loader)
        test_acc = evaluate_accuracy_multi_regression(trained_policy, test_loader)
        tests.append(
            {
                "dataset": dataset_name,
                "method": "NLL",
                "beta": np.nan,
                "rep": rep,
                "test_accuracy": test_acc,
            }
        )

    for beta in beta_list:
        U = torch.eye(C) * float(beta)
        reward = R.OneHotMahalanobis(U, num_classes=C)  # reward

        for rep in range(n_experiments):
            #policy = Policy.CNNClassifier(D_in, C)
            #policy=Policy.ResNet18Vanilla(num_classes=C, in_channels=D_in)
            policy = Policy.MulticlassLogisticRegression(D_in, C)

            loss_fn = L.PO_Entropy_Classification(
                reward_fn=reward,
                n_generations=50,
                use_rsample=False,
                reward_transform="none",
                entropy_weight=entropy_weight,
            )

            beta_trained_policy, train_metrics, val_metrics, _, _ = train_single_policy(
                policy=policy,
                train_dataloader=train_loader,
                val_dataloader=val_loader,
                loss_function=loss_fn,
                n_updates=n_updates,
                learning_rate=learning_rate,
                wandb_run=None,
                tensorboard_writer=None,
                logger=None,
                early_stopping_patience=n_updates, device=torch.device("cpu"))
            

            df_tr = (
                pd.DataFrame(train_metrics)
                .reset_index()
                .rename(columns={"index": "epoch"})
            )
            df_tr["split"] = "train"
            df_tr["method"] = "PO_Entropy"
            df_tr["beta"] = beta
            df_tr["rep"] = rep
            df_val = (
                pd.DataFrame(val_metrics)
                .reset_index()
                .rename(columns={"index": "epoch"})
            )
            df_val["split"] = "val"
            df_val["method"] = "PO_Entropy"
            df_val["beta"] = beta
            df_val["rep"] = rep
            curves += [df_tr, df_val]

            # test_acc = evaluate_accuracy(policy, test_loader)
            test_acc = evaluate_accuracy_multi_regression(
                beta_trained_policy, test_loader
            )
            tests.append(
                {
                    "dataset": dataset_name,
                    "method": "PO_Entropy",
                    "beta": beta,
                    "rep": rep,
                    "test_accuracy": test_acc,
                }
            )

    curves_df = pd.concat(curves, ignore_index=True)
    tests_df = pd.DataFrame(tests)
    curves_df["is_beta_star"] = curves_df["beta"].apply(
        lambda b: isinstance(b, float) and abs(b - beta_star) < 1e-12
    )
    tests_df["is_beta_star"] = tests_df["beta"].apply(
        lambda b: isinstance(b, float) and abs(b - beta_star) < 1e-12
    )
    return curves_df, tests_df, beta_star

In [38]:
import numpy as np
import matplotlib.pyplot as plt

def plot_roc_curve_for_policy(
    policy,
    data_loader,
    device="cpu",
    title="ROC curve",
    save_path="roc_curve.pdf",
    mark_threshold=0.5  # set to None to disable the marker
):
    # Use your existing evaluator
    roc_res = evaluate_roc_auc_binary(policy, data_loader, device=device)
    fpr, tpr, thresholds, auc = roc_res["fpr"], roc_res["tpr"], roc_res["thresholds"], roc_res["auc"]

    # Basic sanity check: binary only
    # (evaluate_roc_auc_binary already implies binary; you can skip this if you trust inputs)
    # If you keep it, you could verify y_true has exactly 2 unique labels.

    plt.figure(figsize=(5, 5))
    plt.plot(fpr, tpr, label=f"ROC (AUC = {auc:.3f})")
    plt.plot([0, 1], [0, 1], "--", linewidth=1, label="Chance")

    # Optionally mark a working threshold (e.g., 0.5) on ROC
    if mark_threshold is not None and len(thresholds) > 0:
        # thresholds may contain non-finite values; ignore them for matching
        mask = np.isfinite(thresholds)
        if np.any(mask):
            idx = np.argmin(np.abs(thresholds[mask] - mark_threshold))
            # Map back to full arrays
            fpr_m = fpr[mask][idx]
            tpr_m = tpr[mask][idx]
            thr_m = thresholds[mask][idx]
            plt.scatter([fpr_m], [tpr_m], zorder=5)
            plt.annotate(f"thr≈{thr_m:.3f}\nTPR={tpr_m:.3f}\nFPR={fpr_m:.3f}",
                         (fpr_m, tpr_m), textcoords="offset points", xytext=(10, -15), fontsize=8)

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    return save_path


In [62]:
# new imports (once)
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix

def run_one_dataset(
    dataset_name: str,
    n_updates: int = 10,
    n_experiments: int = 5,
    batch_size: int = 128,
    learning_rate: float = 1e-4,
    entropy_weight: float = 1e-3,
    type_of_data : str = "tabular",
    corrupted_inputs: bool = False,
):
    device = torch.device("cpu")  # or torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if type_of_data=="tabular":
        print(f"Running on tabular dataset: {dataset_name}")
        train_loader, val_loader, test_loader, meta = load_uci(
            dataset=dataset_name, batch_size=batch_size, standardize=True, impute_missing=True)
        C = meta["num_classes"]
        D_in = meta["input_dim"]

    elif type_of_data=="image" and dataset_name=="mnist":
        print(f"Running on {dataset_name}")
        train_loader, val_loader, test_loader, meta = load_mnist_dataset(
            batch_size=batch_size
        )
        C = meta["num_classes"]
        D_in = meta["in_channels"]

    elif type_of_data=="image" and dataset_name=="cifar10n":
        human_labels_np_path = "/Users/gsinger/Reinforcment_Learning_Huawei/nll_to_po/notebook/CIFAR_NOISY_DATA/CIFAR-10_human.npy"
        train_loader, val_loader, test_loader, meta = load_cifar10n_dataset(
            batch_size=128,
            data_root="./data",
            label_key="random_label2",
            val_fraction=0.2,
            seed=0,
            human_labels_np_path=human_labels_np_path,
        )
        C = meta["num_classes"]
        D_in = meta["in_channels"]

    print('finished to load datasets')
    beta_star = beta_star_from_data(
        train_loader, entropy_weight=entropy_weight, num_classes=C
    )
    print(f"Estimated beta_star: {beta_star:.4e}")
    beta_list = [1, beta_star]

    curves = []
    tests = []

    for rep in range(n_experiments):
        if type_of_data=="tabular":
            policy = Policy.MulticlassLogisticRegression(D_in, C)
        elif type_of_data=="image":
            policy = Policy.ResNet18Vanilla(num_classes=C, in_channels=D_in)

        loss_fn = L.NLL_Classification()
        print("start training NLL")
        trained_policy, train_metrics, val_metrics, _, _ = train_single_policy(
            policy=policy,
            train_dataloader=train_loader,
            val_dataloader=val_loader,
            loss_function=loss_fn,
            n_updates=n_updates,
            learning_rate=learning_rate,
            wandb_run=None,
            tensorboard_writer=None,
            logger=None,
            scheduler_patience=20,
            early_stopping_patience=n_updates,
            device=device,
        )
        #plot_roc_curve_for_policy(trained_policy,test_loader, save_path="roc_curve_nll.pdf", device=str(device), title="ROC Curve for NLL-trained policy", mark_threshold=0.5)
        

        df_tr = pd.DataFrame(train_metrics).reset_index().rename(columns={"index": "epoch"})
        df_tr["split"] = "train"; df_tr["method"] = "NLL"; df_tr["beta"] = np.nan; df_tr["rep"] = rep
        df_val = pd.DataFrame(val_metrics).reset_index().rename(columns={"index": "epoch"})
        df_val["split"] = "val"; df_val["method"] = "NLL"; df_val["beta"] = np.nan; df_val["rep"] = rep
        curves += [df_tr, df_val]

        # --- Test metrics ---
        test_acc = evaluate_accuracy_multi_regression(trained_policy, test_loader)

        # If binary, add TPR/TNR & ROC-AUC via sklearn helpers
        if C == 2:
            thr_metrics = evaluate_tpr_tnr_binary(trained_policy, test_loader, threshold=0.5, device=str(device))
            roc_res     = evaluate_roc_auc_binary(trained_policy, test_loader, device=str(device))
            test_tpr    = thr_metrics["TPR"]
            test_fnr    = thr_metrics["FNR"]
            test_tnr    = thr_metrics["TNR"]
            test_balacc = thr_metrics["BAL_ACC"]
            test_auc    = roc_res["auc"]
        else: #inutile ici!!!
            test_tpr = test_tnr = test_balacc = test_auc = np.nan

        tests.append(
            {
                "dataset": dataset_name,
                "method": "NLL",
                "beta": np.nan,
                "rep": rep,
                "test_accuracy": test_acc,
                "test_tpr": test_tpr,
                "test_fnr" : test_fnr,
                "test_tnr": test_tnr,
                "test_bal_acc": test_balacc,
                "test_auc": test_auc,
            }
        )

    # ----------- PO_Entropy -----------
    for beta in beta_list:
        U = torch.eye(C) * float(beta)
        reward = R.OneHotMahalanobis(U, num_classes=C)

        for rep in range(n_experiments):
            if type_of_data=="tabular":
                policy = Policy.MulticlassLogisticRegression(D_in, C)
            else:
                policy = Policy.ResNet18Vanilla(num_classes=C, in_channels=D_in)

            loss_fn = L.PO_Entropy_Classification(
                reward_fn=reward,
                n_generations=50,
                use_rsample=False,
                reward_transform="none",
                entropy_weight=entropy_weight,
            )

            beta_trained_policy, train_metrics, val_metrics, _, _ = train_single_policy(
                policy=policy,
                train_dataloader=train_loader,
                val_dataloader=val_loader,
                loss_function=loss_fn,
                n_updates=n_updates,
                learning_rate=learning_rate,
                wandb_run=None,
                tensorboard_writer=None,
                logger=None,
                early_stopping_patience=n_updates,
                device=device,
            )
            #plot_roc_curve_for_policy(beta_trained_policy,test_loader, save_path=f"roc_curve_po_entropy_beta{beta:.2e}.pdf", device=str(device), title=f"ROC Curve for PO_Entropy-trained policy (beta={beta:.2e})", mark_threshold=0.5)

            df_tr = pd.DataFrame(train_metrics).reset_index().rename(columns={"index": "epoch"})
            df_tr["split"] = "train"; df_tr["method"] = "PO_Entropy"; df_tr["beta"] = beta; df_tr["rep"] = rep
            df_val = pd.DataFrame(val_metrics).reset_index().rename(columns={"index": "epoch"})
            df_val["split"] = "val"; df_val["method"] = "PO_Entropy"; df_val["beta"] = beta; df_val["rep"] = rep
            curves += [df_tr, df_val]

            # --- Test metrics ---
            test_acc = evaluate_accuracy_multi_regression(beta_trained_policy, test_loader)

            if C == 2:
                thr_metrics = evaluate_tpr_tnr_binary(beta_trained_policy, test_loader, threshold=0.5, device=str(device))
                roc_res     = evaluate_roc_auc_binary(beta_trained_policy, test_loader, device=str(device))
                test_tpr    = thr_metrics["TPR"]
                test_tnr    = thr_metrics["TNR"]
                test_fnr    = thr_metrics["FNR"]
                test_balacc = thr_metrics["BAL_ACC"]
                test_auc    = roc_res["auc"]
            else:
                test_tpr = test_tnr = test_balacc = test_auc = np.nan

            tests.append(
                {
                    "dataset": dataset_name,
                    "method": "PO_Entropy",
                    "beta": beta,
                    "rep": rep,
                    "test_accuracy": test_acc,
                    "test_tpr": test_tpr,
                    "test_tnr": test_tnr,
                    "test_fnr" : test_fnr,
                    "test_bal_acc": test_balacc,
                    "test_auc": test_auc,
                }
            )

    curves_df = pd.concat(curves, ignore_index=True)
    tests_df = pd.DataFrame(tests)

    curves_df["is_beta_star"] = curves_df["beta"].apply(
        lambda b: isinstance(b, float) and abs(b - beta_star) < 1e-12
    )
    tests_df["is_beta_star"] = tests_df["beta"].apply(
        lambda b: isinstance(b, float) and abs(b - beta_star) < 1e-12
    )
    return curves_df, tests_df, beta_star


In [13]:
def plot_curves_for_dataset(
    curves_df: pd.DataFrame, dataset_name: str, beta_star: float
):
    sns.set_style("whitegrid")
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

    # helpers
    def make_label(row):
        if row["method"] == "NLL":
            return "NLL"
        if pd.isna(row["beta"]):
            return "PO (β=NA)"
        if abs(row["beta"] - beta_star) < 1e-12:
            return "PO (β*)"
        return f"PO (β={row['beta']:.3g})"

    curves_df = curves_df.copy()
    curves_df["label"] = curves_df.apply(make_label, axis=1)

    # colors: NLL blue, β* black, others red
    palette_map = {}
    for lab in curves_df["label"].unique():
        if lab == "NLL":
            palette_map[lab] = "#1f77b4"
        elif lab == "PO (β*)":
            palette_map[lab] = "black"
        else:
            palette_map[lab] = "red"

    # TRAIN
    sub = curves_df[curves_df["split"] == "train"]
    sns.lineplot(
        data=sub,
        x="epoch",
        y="accuracy",
        hue="label",
        errorbar=("ci", 95),
        ax=ax[0],
        palette=palette_map,
        legend=False,
    )
    ax[0].set_title(f"{dataset_name}: Train accuracy vs epoch")
    ax[0].set_xlabel("epoch")
    ax[0].set_ylabel("accuracy")

    # VAL
    sub = curves_df[curves_df["split"] == "val"]
    sns.lineplot(
        data=sub,
        x="epoch",
        y="accuracy",
        hue="label",
        errorbar=("ci", 95),
        ax=ax[1],
        palette=palette_map,
        legend=True,
    )
    ax[1].set_title(f"{dataset_name}: Val accuracy vs epoch")
    ax[1].set_xlabel("epoch")
    ax[1].set_ylabel("accuracy")
    ax[1].legend(title="method", frameon=False, loc="lower right")

    plt.tight_layout()
    plt.show()

In [14]:
def show_test_table(tests_df: pd.DataFrame, dataset_name: str):
    keep = (tests_df["method"] == "PO_Entropy") | (tests_df["method"] == "NLL")
    df = tests_df[keep].copy()

    grouped = (
        df.groupby(["dataset", "method", "is_beta_star"], dropna=False)["test_accuracy"]
        .agg(["mean", "std", "count"])
        .reset_index()
    )

    def beta_label(row):
        if row["method"] == "NLL":
            return "—"
        return "β*" if row["is_beta_star"] else "β=1"

    grouped["beta_label"] = grouped.apply(beta_label, axis=1)
    grouped = grouped[["dataset", "method", "beta_label", "mean", "std", "count"]]
    grouped = grouped[grouped["dataset"] == dataset_name]

    print(f"Test accuracy summary — {dataset_name}")
    display(grouped.style.format({"mean": "{:.4f}", "std": "{:.4f}"}))

In [44]:
import pandas as pd
import numpy as np

def show_test_table(tests_df: pd.DataFrame, dataset_name: str, methods=("PO_Entropy", "NLL"), beta_sig=4, show_pivot=False):
    """
    - Agrège test_accuracy par (dataset, method, beta).
    - Gère de nombreuses valeurs de beta.
    - Ajoute un label lisible pour beta: 'β*' (si is_beta_star), 'β=...' sinon, '—' pour NLL (beta NaN).
    - Option show_pivot=True: affiche un tableau compact (index=beta_label, colonnes=method, valeurs=mean±std).
    """
    # Filtre méthodes et dataset
    df = tests_df[tests_df["method"].isin(methods)].copy()
    df = df[df["dataset"] == dataset_name].copy()

    # S'assure que la colonne beta existe (au cas où)
    if "beta" not in df.columns:
        df["beta"] = pd.NA
    if "is_beta_star" not in df.columns:
        df["is_beta_star"] = False

    # Agrégation par beta
    grouped = (
        df.groupby(["dataset", "method", "beta"], dropna=False)["test_accuracy"]
          .agg(mean="mean", std="std", count="count")
          .reset_index()
    )

    # Marqueurs β*
    star = (
        df.groupby(["dataset", "method", "beta"], dropna=False)["is_beta_star"]
          .any().reset_index(name="is_beta_star")
    )
    grouped = grouped.merge(star, on=["dataset", "method", "beta"], how="left")

    # Label lisible pour beta
    def beta_label_row(row):
        if pd.isna(row["beta"]):
            return "—"  # typiquement NLL où beta est NaN
        if row["is_beta_star"]:
            return "β*"
        # format compact
        try:
            return f"β={float(row['beta']):.{beta_sig}g}"
        except Exception:
            return f"β={row['beta']}"

    grouped["beta_label"] = grouped.apply(beta_label_row, axis=1)

    # Colonnes finales + tri (placer NaN à la fin)
    grouped = grouped[["dataset", "method", "beta", "beta_label", "mean", "std", "count"]]
    grouped = grouped.sort_values(
        by=["method", "beta"],
        key=lambda s: s.map(lambda x: np.inf if (isinstance(x, float) and np.isnan(x)) else x) if s.name == "beta" else s
    )

    print(f"Test accuracy summary — {dataset_name}")

    if show_pivot:
        # Tableau compact: lignes = beta_label, colonnes = method, valeurs = "mean ± std"
        pivot = grouped.copy()
        pivot["mean±std"] = pivot.apply(lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1)
        pivot = pivot.pivot(index="beta_label", columns="method", values="mean±std")
        display(pivot)  # type: ignore  # si tu es en notebook
    else:
        # Affichage standard
        display(grouped.style.format({"mean": "{:.4f}", "std": "{:.4f}"}))  # type: ignore

    return grouped


In [45]:
import pandas as pd
import numpy as np

def show_test_table(
    tests_df: pd.DataFrame,
    dataset_name: str,
    methods=("PO_Entropy", "NLL"),
    beta_sig: int = 4,
    show_pivot: bool = False,
    metrics=("test_accuracy", "test_bal_acc", "test_auc", "test_tpr", "test_tnr", "test_fnr"),
):
    """
    Summarize test metrics per (dataset, method, beta).

    - Aggregates mean/std over repetitions for each metric in `metrics` (if present).
    - Adds readable beta labels:
        'β*' if is_beta_star,
        'β=...' for numeric betas,
        '—' for NaN beta (e.g., NLL).
    - If show_pivot=True: shows a compact pivot with rows = beta_label,
      columns = (metric, method), values = "mean ± std".

    Returns:
        grouped (pd.DataFrame): numeric table with columns like:
            ['dataset','method','beta','beta_label','count',
             'test_accuracy_mean','test_accuracy_std', ...]
    """
    # --- Filter dataset & methods
    df = tests_df[tests_df["method"].isin(methods)].copy()
    df = df[df["dataset"] == dataset_name].copy()
    if df.empty:
        print(f"No rows for dataset '{dataset_name}' and methods {methods}.")
        return pd.DataFrame()

    # Ensure columns exist
    if "beta" not in df.columns:
        df["beta"] = pd.NA
    if "is_beta_star" not in df.columns:
        df["is_beta_star"] = False

    # Keep only metrics that exist in df
    metric_cols = [m for m in metrics if m in df.columns]
    if not metric_cols:
        # fallback to accuracy if nothing else is present
        if "test_accuracy" in df.columns:
            metric_cols = ["test_accuracy"]
        else:
            print("No requested metric columns found in tests_df.")
            return pd.DataFrame()

    # --- Aggregate: mean/std per (dataset, method, beta), + count
    agg_dict = {m: ["mean", "std"] for m in metric_cols}
    grouped = (
        df.groupby(["dataset", "method", "beta"], dropna=False)
          .agg(agg_dict)
          .reset_index()
    )
    # flat columns: metric_mean / metric_std
    grouped.columns = [
        "_".join([c[0], c[1]]).strip("_")
        if isinstance(c, tuple) and c[1] != ""
        else (c if not isinstance(c, tuple) else c[0])
        for c in grouped.columns
    ]
    # add count (number of rows / reps per group)
    counts = (
        df.groupby(["dataset", "method", "beta"], dropna=False)
          .size()
          .reset_index(name="count")
    )
    grouped = grouped.merge(counts, on=["dataset", "method", "beta"], how="left")

    # Merge β* flags
    star = (
        df.groupby(["dataset", "method", "beta"], dropna=False)["is_beta_star"]
          .any().reset_index(name="is_beta_star")
    )
    grouped = grouped.merge(star, on=["dataset", "method", "beta"], how="left")

    # --- Beta label pretty printer
    def beta_label_row(row):
        if pd.isna(row["beta"]):
            return "—"  # e.g., NLL
        if row["is_beta_star"]:
            return "β*"
        try:
            return f"β={float(row['beta']):.{beta_sig}g}"
        except Exception:
            return f"β={row['beta']}"

    grouped["beta_label"] = grouped.apply(beta_label_row, axis=1)

    # Sort: NaN betas last per method
    grouped = grouped.sort_values(
        by=["method", "beta"],
        key=lambda s: (
            s.map(lambda x: np.inf if (isinstance(x, float) and np.isnan(x)) else x)
            if s.name == "beta" else s
        )
    )

    # Column order: dataset, method, beta, beta_label, count, then metric cols
    metric_mean_cols = [f"{m}_mean" for m in metric_cols if f"{m}_mean" in grouped.columns]
    metric_std_cols  = [f"{m}_std"  for m in metric_cols if f"{m}_std"  in grouped.columns]
    ordered_cols = ["dataset", "method", "beta", "beta_label", "count"] + metric_mean_cols + metric_std_cols
    grouped = grouped[ordered_cols]

    print(f"Test metrics summary — {dataset_name}")

    # --- Display
    try:
        from IPython.display import display  # noqa: F401
        have_display = True
    except Exception:
        have_display = False

    if show_pivot:
        # Build a long dataframe with "mean ± std" per metric for pivoting
        long_rows = []
        for _, r in grouped.iterrows():
            for m in metric_cols:
                m_mean = r.get(f"{m}_mean", np.nan)
                m_std  = r.get(f"{m}_std",  np.nan)
                long_rows.append({
                    "beta_label": r["beta_label"],
                    "method": r["method"],
                    "metric": m,
                    "mean±std": f"{m_mean:.4f} ± {m_std:.4f}" if pd.notna(m_mean) and pd.notna(m_std) else "—",
                })
        long_df = pd.DataFrame(long_rows)
        if long_df.empty:
            if have_display:
                display(grouped.style.format(precision=4))
            else:
                print(grouped.to_string(index=False))
            return grouped

        pivot = long_df.pivot(index="beta_label", columns=["metric", "method"], values="mean±std")
        # Sort metrics in the requested order
        # (columns is a MultiIndex: level 0 = metric, level 1 = method)
        pivot = pivot.reindex(index=sorted(pivot.index), columns=pivot.columns.reindex(level=0, labels=metric_cols)[0])
        if have_display:
            display(pivot)
        else:
            print(pivot.to_string())
    else:
        # Pretty numeric table
        fmt = {col: "{:.4f}" for col in metric_mean_cols + metric_std_cols}
        if have_display:
            display(grouped.style.format(fmt))
        else:
            print(grouped.to_string(index=False))

    return grouped


In [17]:
"aps_failure": 421,           # APS Failure at Scania Trucks
    "secom": 179,                 # Semiconductor Manufacturing (SECOM)
    "bank_marketing": 222,        # Bank Marketing
    "australian_credit": 143,     # Statlog Australian Credit Approval
    "magic_gamma": 159,           # OK MAGIC Gamma Telescope
    "ionosphere": 52,             # Ionosphere
    "wilt": 285,                  # Wilt (diseased trees)
    "higgs": 280,                 # HIGGS
    "spambase": 94,               # (optionnel) Spambase texte
}     


SyntaxError: unmatched '}' (2616059717.py, line 10)

In [98]:
datasets = ["credit_default"] #"Yeast"

all_curves = []
all_tests = []
#or R
for ds in datasets:
    curves_df, tests_df, bstar = run_one_dataset(
        dataset_name=ds,
        n_updates=30,
        n_experiments=3,
        batch_size=128,
        learning_rate=4*1e-2, #4
        entropy_weight=10, 
        type_of_data= "tabular"
    )
    plot_curves_for_dataset(curves_df, ds, bstar)
    show_test_table(tests_df, ds)

    curves_df["dataset"] = ds
    tests_df["dataset"] = ds
    all_curves.append(curves_df)
    all_tests.append(tests_df)

df_curves_all = pd.concat(all_curves, ignore_index=True)
df_tests_all = pd.concat(all_tests, ignore_index=True)

Running on tabular dataset: credit_default
finished to load datasets
in beta_star_from_data
in estimate_trace_sigma_onehot
finished counts
len(dataset) = 19200
len(loader)  = 150
Estimated beta_star: 2.9023e+01
start training NLL


Training epochs: 100%|██████████| 30/30 [00:05<00:00,  5.96it/s]


start training NLL


Training epochs:   3%|▎         | 1/30 [00:00<00:07,  3.68it/s]


KeyboardInterrupt: 

In [30]:
df_curves_a.head()

,epoch,NLL,accuracy,entropy,grad_norm,split,method,beta,rep,pg_loss,total_loss,is_beta_star,dataset
0,0,0.516036,0.791771,0.427720,0.407568,train,NLL,NaN,0,NaN,NaN,False,credit_default
1,1,0.509276,0.796094,0.425728,0.402769,train,NLL,NaN,0,NaN,NaN,False,credit_default
2,2,0.505518,0.798177,0.427859,0.396207,train,NLL,NaN,0,NaN,NaN,False,credit_default
3,3,0.510660,0.795312,0.425738,0.397637,train,NLL,NaN,0,NaN,NaN,False,credit_default
4,4,0.509086,0.796354,0.422769,0.392393,train,NLL,NaN,0,NaN,NaN,False,credit_default


In [31]:
df_curves_all.to_csv("/Users/gsinger/Reinforcment_Learning_Huawei/nll_to_po/notebook/BISFULL_CREDIT_CARD_curves_all.csv", index=False)

In [32]:
df_tests_all.to_csv("/Users/gsinger/Reinforcment_Learning_Huawei/nll_to_po/notebook/BISFULL_CREDIT_CARD_tests_all.csv", index=False)

In [92]:
show_test_table(df_tests_all,
    "credit_default", metrics=("test_accuracy","test_auc"))

Test metrics summary — credit_default


,dataset,method,beta,beta_label,count,test_accuracy_mean,test_auc_mean,test_accuracy_std,test_auc_std
0,credit_default,NLL,nan,—,2,0.7979,0.7054,0.0110,0.0071
1,credit_default,PO_Entropy,1.000000,β=1,2,0.7553,0.5771,0.0153,0.0795
2,credit_default,PO_Entropy,29.022795,β*,2,0.8203,0.7133,0.0005,0.0001


,dataset,method,beta,beta_label,count,test_accuracy_mean,test_auc_mean,test_accuracy_std,test_auc_std
0,credit_default,NLL,NaN,—,2,0.797917,0.705385,0.010960,0.007148
1,credit_default,PO_Entropy,1.000000,β=1,2,0.755333,0.577082,0.015321,0.079457
2,credit_default,PO_Entropy,29.022795,β*,2,0.820333,0.713258,0.000471,0.000149


In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def plot_val_curves_with_ci(
    csv_path: str,
    metric: str = "accuracy",
    split: str = "val",
    save_path: str = "val_accuracy_curve.png"
):
    """
    - Lit un CSV avec colonnes: ['epoch','split','method','beta','rep','is_beta_star', <metrics...>]
    - Filtre sur `split` ('train' ou 'val')
    - Agrège sur les répétitions: moyenne & écart-type par (method, beta, is_beta_star, epoch)
    - Trace NLL + une courbe par config PG, avec ruban ±1 std
    """
    df = pd.read_csv(csv_path)
    df = df[df["split"] == split].copy()
    if df.empty:
        raise ValueError(f"No rows for split='{split}' in {csv_path}")

    group_keys = ["method", "beta", "is_beta_star", "epoch"]
    agg_mean = df.groupby(group_keys, dropna=False)[metric].mean().reset_index()
    agg_std  = df.groupby(group_keys, dropna=False)[metric].std(ddof=0).reset_index()
    agg = pd.merge(agg_mean, agg_std, on=group_keys, suffixes=("_mean", "_std"))

    # Assure types numériques
    agg[f"{metric}_mean"] = pd.to_numeric(agg[f"{metric}_mean"], errors="coerce")
    agg[f"{metric}_std"]  = pd.to_numeric(agg[f"{metric}_std"], errors="coerce").fillna(0.0)

    # ---------- Plot ----------
    fig, ax = plt.subplots(1, 1, figsize=(5, 5))

    nll = agg[agg["method"] == "NLL"].sort_values("epoch")
    pg  = agg[agg["method"] != "NLL"]

    # NLL
    if not nll.empty:
        ax.plot(nll["epoch"].values, nll[f"{metric}_mean"].values, label="NLL", color="red")
        if np.any(nll[f"{metric}_std"].values > 0):
            ax.fill_between(
                nll["epoch"].values,
                (nll[f"{metric}_mean"] - nll[f"{metric}_std"]).values,
                (nll[f"{metric}_mean"] + nll[f"{metric}_std"]).values,
                alpha=0.2, color="red"
            )

    # PG courbes
    for (b, is_star), g in pg.groupby(["beta", "is_beta_star"], dropna=False):
        g = g.sort_values("epoch")
        if is_star:
            label = r"PG($U^\star$)"
        else:
            label = r"PG($U = I_n$)"
        ax.plot(g["epoch"].values, g[f"{metric}_mean"].values, label=label)
        if np.any(g[f"{metric}_std"].values > 0):
            ax.fill_between(
                g["epoch"].values,
                (g[f"{metric}_mean"] - g[f"{metric}_std"]).values,
                (g[f"{metric}_mean"] + g[f"{metric}_std"]).values,
                alpha=0.2,
            )

    # Titre et axes
    if metric.lower() in ["accuracy", "acc"]:
        ax.set_title("Validation Accuracy")
        ax.set_ylim(0.0, 1.0)
        ax.set_xlim(0, 30)
        ax.set_ylabel("Accuracy")
    if metric.lower() in ["NLL"]:
        ax.set_title("Negative-Log-Likelihood")
        ax.set_ylim(0.0, 0.5)
        ax.set_xlim(0, 30)
        ax.set_ylabel("Negative-Log-Likelihood")
    else:
        ax.set_title(f"{split.capitalize()} {metric.capitalize()}")
        ax.set_ylabel(metric.capitalize())

    ax.set_xlabel("Epoch")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="lower right")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return save_path


# Exemple d'appel
out_path = plot_val_curves_with_ci(
    "/Users/gsinger/Reinforcment_Learning_Huawei/nll_to_po/notebook/2_50_lr_0_01_lambda01_CIFAR_df_curves_all.csv",
    metric="accuracy",
    split="val",
    save_path="CIFAR_Acc_curve.png"
)
print("Saved:", out_path)


Saved: CIFAR_Acc_curve.png


In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def plot_val_curves_with_ci(
    csv_path: str,
    metric: str = "accuracy",
    split: str = "val",
    save_path: str = "val_accuracy_curve.png"
):
    df = pd.read_csv(csv_path)
    df = df[df["split"] == split].copy()
    if df.empty:
        raise ValueError(f"No rows for split='{split}' in {csv_path}")

    group_keys = ["method", "beta", "is_beta_star", "epoch"]
    agg_mean = df.groupby(group_keys, dropna=False)[metric].mean().reset_index()
    agg_std  = df.groupby(group_keys, dropna=False)[metric].std(ddof=0).reset_index()
    agg = pd.merge(agg_mean, agg_std, on=group_keys, suffixes=("_mean", "_std"))

    agg[f"{metric}_mean"] = pd.to_numeric(agg[f"{metric}_mean"], errors="coerce")
    agg[f"{metric}_std"]  = pd.to_numeric(agg[f"{metric}_std"], errors="coerce").fillna(0.0)

    fig, ax = plt.subplots(1, 1, figsize=(5, 5))

    nll = agg[agg["method"] == "NLL"].sort_values("epoch")
    pg  = agg[agg["method"] != "NLL"]

    if not nll.empty:
        ax.plot(nll["epoch"].values, nll[f"{metric}_mean"].values, label="NLL", color="red")
        if np.any(nll[f"{metric}_std"].values > 0):
            ax.fill_between(
                nll["epoch"].values,
                (nll[f"{metric}_mean"] - nll[f"{metric}_std"]).values,
                (nll[f"{metric}_mean"] + nll[f"{metric}_std"]).values,
                alpha=0.2, color="red"
            )

    for (b, is_star), g in pg.groupby(["beta", "is_beta_star"], dropna=False):
        g = g.sort_values("epoch")
        label = r"PG($U^\star$)" if is_star else r"PG($I_n$)"
        ax.plot(g["epoch"].values, g[f"{metric}_mean"].values, label=label)
        if np.any(g[f"{metric}_std"].values > 0):
            ax.fill_between(
                g["epoch"].values,
                (g[f"{metric}_mean"] - g[f"{metric}_std"]).values,
                (g[f"{metric}_mean"] + g[f"{metric}_std"]).values,
                alpha=0.2,
            )

    # Titles and axis ranges
    if metric.lower() in ["accuracy", "acc"]:
        ax.set_title("Validation Accuracy")
        ax.set_ylim(0.5, 1.0)
        ax.set_xlim(0, 30)
        ax.set_ylabel("Accuracy")
    elif metric.lower() in ["nll"]:
        ax.set_title("Negative-Log-Likelihood")
        ax.set_ylim(0.0, 300)
        ax.set_xlim(0, 50)
        ax.set_ylabel("Negative-Log-Likelihood")
    else:
        ax.set_title(f"{split.capitalize()} {metric.capitalize()}")
        ax.set_ylabel(metric.capitalize())

    ax.set_xlabel("Epoch")
    ax.grid(True, alpha=0.3)

    # Legend below the plot, centered, framed
    ax.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.20),   # position below
        ncol=3,                        # number of columns
        frameon=True,                  # draw a box
        fancybox=True,                 # rounded corners
        shadow=False                   # add shadow if you like
    )

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return save_path


# Example call
out_path = plot_val_curves_with_ci(
    "/Users/gsinger/Reinforcment_Learning_Huawei/nll_to_po/notebook/BISFULL_CREDIT_CARD_curves_all.csv",
    metric="accuracy",
    split="val",
    save_path="CREDIT_CARD_L_curve.pdf"
)
print("Saved:", out_path)


Saved: CREDIT_CARD_L_curve.pdf


In [34]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def _beta_legend(beta, is_star):
    # 3 chiffres significatifs pour lisibilité
    if beta is None or (isinstance(beta, float) and np.isnan(beta)):
        return r"PG($\beta$=NA)"
    fmt = f"{float(beta):.3g}"
    if is_star:
        return rf"PG($\beta^\star\!\approx\!{fmt}$)"
    if np.isclose(float(beta), 1.0):
        return r"PG($\beta=1$)"
    return rf"PG($\beta={fmt}$)"

def plot_val_curves_with_ci(
    csv_path: str,
    metric: str = "accuracy",
    split: str = "val",
    save_path: str = "val_accuracy_curve.png"
):
    df = pd.read_csv(csv_path)
    df = df[df["split"] == split].copy()
    if df.empty:
        raise ValueError(f"No rows for split='{split}' in {csv_path}")

    group_keys = ["method", "beta", "is_beta_star", "epoch"]
    agg_mean = df.groupby(group_keys, dropna=False)[metric].mean().reset_index()
    agg_std  = df.groupby(group_keys, dropna=False)[metric].std(ddof=0).reset_index()
    agg = pd.merge(agg_mean, agg_std, on=group_keys, suffixes=("_mean", "_std"))

    agg[f"{metric}_mean"] = pd.to_numeric(agg[f"{metric}_mean"], errors="coerce")
    agg[f"{metric}_std"]  = pd.to_numeric(agg[f"{metric}_std"], errors="coerce").fillna(0.0)

    fig, ax = plt.subplots(1, 1, figsize=(5, 5))

    # NLL (courbe rouge + bande de confiance)
    nll = agg[agg["method"] == "NLL"].sort_values("epoch")
    pg  = agg[agg["method"] != "NLL"]

    if not nll.empty:
        ax.plot(nll["epoch"].values, nll[f"{metric}_mean"].values, label="NLL", color="red")
        if np.any(nll[f"{metric}_std"].values > 0):
            ax.fill_between(
                nll["epoch"].values,
                (nll[f"{metric}_mean"] - nll[f"{metric}_std"]).values,
                (nll[f"{metric}_mean"] + nll[f"{metric}_std"]).values,
                alpha=0.2, color="red"
            )

    # PG: une courbe par valeur de beta (y compris β* et β=1)
    for (b, is_star), g in pg.groupby(["beta", "is_beta_star"], dropna=False):
        g = g.sort_values("epoch")
        label = _beta_legend(b, bool(is_star))
        ax.plot(g["epoch"].values, g[f"{metric}_mean"].values, label=label)
        if np.any(g[f"{metric}_std"].values > 0):
            ax.fill_between(
                g["epoch"].values,
                (g[f"{metric}_mean"] - g[f"{metric}_std"]).values,
                (g[f"{metric}_mean"] + g[f"{metric}_std"]).values,
                alpha=0.2,
            )

    # Titres et axes
    if metric.lower() in ["accuracy", "acc"]:
        ax.set_title("Validation Accuracy")
        # Limites dynamiques (évite de couper des courbes < 0.9)
        ymin = max(0.0, (agg[f"{metric}_mean"] - agg[f"{metric}_std"]).min())
        ymax = min(1.0, (agg[f"{metric}_mean"] + agg[f"{metric}_std"]).max())
        m = 0.02
        ax.set_ylim(ymin - m, ymax + m)
        ax.set_xlim(df["epoch"].min(), df["epoch"].max())
        ax.set_ylabel("Accuracy")
    elif metric.lower() in ["nll"]:
        ax.set_title("Negative-Log-Likelihood")
        ax.set_xlim(df["epoch"].min(), df["epoch"].max())
        ax.set_ylabel("Negative-Log-Likelihood")
    else:
        ax.set_title(f"{split.capitalize()} {metric.capitalize()}")
        ax.set_xlim(df["epoch"].min(), df["epoch"].max())
        ax.set_ylabel(metric.capitalize())

    ax.set_xlabel("Epoch")
    ax.grid(True, alpha=0.3)

    # Légende sous la figure
    ax.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.20),
        ncol=4,            # 3 β + NLL
        frameon=True,
        fancybox=True,
    )

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return save_path
# Example call
out_path = plot_val_curves_with_ci(
    "/Users/gsinger/Reinforcment_Learning_Huawei/nll_to_po/notebook/BISFULL_CREDIT_CARD_curves_all.csv",
    metric="accuracy",
    split="val",
    save_path="CREDIT_CARD_L_curve.pdf"
)
print("Saved:", out_path)


Saved: CREDIT_CARD_L_curve.pdf
